SOURCE TO TARGET (ROW COUNT VALIDATION)

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA bronze;
SELECT count(*) as bronze_count
FROM bronze.customers_raw;

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA silver;
SELECT count(*) as silver_count
FROM silver.customers_clean;

DATA QUALITY VALIDATION

In [0]:
%sql
-- DUPLICATE CHECK (Bronze)

USE CATALOG retail_sales_catalog;
USE SCHEMA bronze;
SELECT 
    COALESCE(CAST(CustomerID AS STRING), 'NULL') AS CustomerID,
    COUNT(*) AS cnt
FROM bronze.customers_raw
GROUP BY CustomerID
HAVING COUNT(*) > 1
ORDER BY cnt DESC;

In [0]:
%sql
-- DUPLICATE CHECK (Silver)

USE CATALOG retail_sales_catalog;
USE SCHEMA silver;
SELECT CustomerID, COUNT(*)
FROM silver.customers_clean
WHERE CustomerID IS NOT NULL
GROUP BY CustomerID
HAVING COUNT(*) > 1;

SCD TYPE 2 VALIDATION

In [0]:
%sql
SELECT CustomerID, COUNT(*)
FROM gold.dim_customer
WHERE IsActive = 1
GROUP BY CustomerID
HAVING COUNT(*) > 1;

FACT TABLE VALIDATION

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA gold;
SELECT COUNT(*) FROM gold.fact_sales;

ARCHIVAL VALIDATION

In [0]:
files = dbutils.fs.ls("s3://retaill-client/retaill-lakehouse/landing/")

customers_files = [f.name for f in files if "customers" in f.name]

print(customers_files)
print("Count:", len(customers_files))

In [0]:
for entity in ["customers", "products", "stores", "sales"]:
    entity_files = [f.name for f in files if entity in f.name]
    
    if len(entity_files) > 1:
        print(f"❌ Archival failed for {entity}")
    else:
        print(f"✔ {entity} is correct")

Check SCD:

In [0]:
%sql
SELECT *
FROM gold.dim_customer
WHERE CustomerID IN (
    SELECT CustomerID
    FROM silver.customers_clean
)
ORDER BY CustomerID, StartDate;

Check Fact:

In [0]:
%sql
SELECT COUNT(*) FROM gold.fact_sales;

Check Archive:

In [0]:
display(dbutils.fs.ls("s3://retaill-client/retaill-lakehouse/archive/"))

### TRANSFORMATION VALIDATION

In [0]:
%sql
-- Validate Email Lowercase Transformation
SELECT 
    'Email Lowercase' as validation_name,
    COUNT(*) as failed_count
FROM retail_sales_catalog.silver.customers_clean
WHERE Email != LOWER(Email) OR Email IS NULL;

In [0]:
%sql
-- Validate Trim Transformations (check for leading/trailing spaces)
SELECT 
    'Trimming Validation' as validation_name,
    SUM(CASE WHEN City != TRIM(City) THEN 1 ELSE 0 END) as city_not_trimmed,
    SUM(CASE WHEN Address != TRIM(Address) THEN 1 ELSE 0 END) as address_not_trimmed,
    SUM(CASE WHEN CustomerName != TRIM(CustomerName) THEN 1 ELSE 0 END) as name_not_trimmed
FROM retail_sales_catalog.silver.customers_clean;

In [0]:
%sql
-- Validate Amount Calculation (Quantity × UnitPrice)
SELECT 
    f.TransactionID,
    f.Quantity,
    p.UnitPrice,
    f.Amount,
    (f.Quantity * p.UnitPrice) as expected_amount,
    CASE 
        WHEN f.Amount = (f.Quantity * p.UnitPrice) THEN 'PASS'
        ELSE 'FAIL'
    END as validation_status
FROM retail_sales_catalog.gold.fact_sales f
JOIN retail_sales_catalog.gold.dim_product p ON f.ProductSK = p.ProductSK
WHERE f.Amount != (f.Quantity * p.UnitPrice)
LIMIT 10;

### NULL VALUE VALIDATION

In [0]:
%sql
-- Check for NULLs in critical columns (Silver)
SELECT 
    'Silver Customers' as layer,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) as null_customer_id,
    SUM(CASE WHEN CustomerName IS NULL THEN 1 ELSE 0 END) as null_customer_name,
    SUM(CASE WHEN Email IS NULL THEN 1 ELSE 0 END) as null_email,
    SUM(CASE WHEN City IS NULL THEN 1 ELSE 0 END) as null_city,
    SUM(CASE WHEN Address IS NULL THEN 1 ELSE 0 END) as null_address
FROM retail_sales_catalog.silver.customers_clean;

In [0]:
%sql
-- Check for NULLs in Fact Table
SELECT 
    'Fact Sales' as table_name,
    SUM(CASE WHEN CustomerSK IS NULL THEN 1 ELSE 0 END) as null_customer_sk,
    SUM(CASE WHEN ProductSK IS NULL THEN 1 ELSE 0 END) as null_product_sk,
    SUM(CASE WHEN StoreSK IS NULL THEN 1 ELSE 0 END) as null_store_sk,
    SUM(CASE WHEN TxnDate IS NULL THEN 1 ELSE 0 END) as null_txn_date,
    SUM(CASE WHEN Quantity IS NULL THEN 1 ELSE 0 END) as null_quantity,
    SUM(CASE WHEN Amount IS NULL THEN 1 ELSE 0 END) as null_amount
FROM retail_sales_catalog.gold.fact_sales;

### REFERENTIAL INTEGRITY VALIDATION

In [0]:
%sql
-- Check for orphaned records in Fact (CustomerSK not in Dim)
SELECT 
    'Orphaned CustomerSK' as validation_name,
    COUNT(DISTINCT f.CustomerSK) as orphaned_count
FROM retail_sales_catalog.gold.fact_sales f
LEFT JOIN retail_sales_catalog.gold.dim_customer c ON f.CustomerSK = c.CustomerSK
WHERE c.CustomerSK IS NULL;

In [0]:
%sql
-- Check for orphaned ProductSK and StoreSK
SELECT 
    'Referential Integrity' as validation_type,
    COUNT(DISTINCT CASE WHEN p.ProductSK IS NULL THEN f.ProductSK END) as orphaned_product_sk,
    COUNT(DISTINCT CASE WHEN s.StoreSK IS NULL THEN f.StoreSK END) as orphaned_store_sk
FROM retail_sales_catalog.gold.fact_sales f
LEFT JOIN retail_sales_catalog.gold.dim_product p ON f.ProductSK = p.ProductSK
LEFT JOIN retail_sales_catalog.gold.dim_store s ON f.StoreSK = s.StoreSK;

### SCD TYPE 2 VALIDATION

In [0]:
%sql
-- Validate SCD Type 2 Date Logic
SELECT 
    'SCD Date Validation' as validation_name,
    SUM(CASE WHEN StartDate > EndDate AND EndDate IS NOT NULL THEN 1 ELSE 0 END) as invalid_date_range,
    SUM(CASE WHEN IsActive = 1 AND EndDate IS NOT NULL THEN 1 ELSE 0 END) as active_with_end_date,
    SUM(CASE WHEN IsActive = 0 AND EndDate IS NULL THEN 1 ELSE 0 END) as inactive_without_end_date
FROM retail_sales_catalog.gold.dim_customer;

In [0]:
%sql
-- Check for overlapping SCD records (same CustomerID active in same time period)
SELECT 
    c1.CustomerID,
    c1.CustomerSK as SK1,
    c2.CustomerSK as SK2,
    c1.StartDate as Start1,
    c1.EndDate as End1,
    c2.StartDate as Start2,
    c2.EndDate as End2
FROM retail_sales_catalog.gold.dim_customer c1
JOIN retail_sales_catalog.gold.dim_customer c2 
    ON c1.CustomerID = c2.CustomerID 
    AND c1.CustomerSK != c2.CustomerSK
WHERE c1.StartDate <= COALESCE(c2.EndDate, CURRENT_DATE)
  AND c2.StartDate <= COALESCE(c1.EndDate, CURRENT_DATE)
LIMIT 10;

### DATA QUALITY VALIDATION

In [0]:
%sql
-- Validate Email Format
SELECT 
    'Email Format Validation' as validation_name,
    COUNT(*) as invalid_email_count
FROM retail_sales_catalog.silver.customers_clean
WHERE Email NOT LIKE '%@%.%' OR Email IS NULL;

In [0]:
%sql
-- Validate Quantity and Amount are positive
SELECT 
    'Data Quality - Fact' as validation_name,
    SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) as negative_quantity_count,
    SUM(CASE WHEN Amount <= 0 THEN 1 ELSE 0 END) as negative_amount_count,
    MIN(Quantity) as min_quantity,
    MIN(Amount) as min_amount
FROM retail_sales_catalog.gold.fact_sales;

### RECONCILIATION SUMMARY

In [0]:
%sql
-- End-to-End Reconciliation Report with Raw Counts
SELECT 
    'Bronze Layer' as stage,
    'Raw (Total)' as description,
    (SELECT COUNT(*) FROM retail_sales_catalog.bronze.customers_raw) as record_count,
    NULL as difference_from_previous
UNION ALL
SELECT 
    'Bronze Layer',
    'Valid (Excluding NULLs)',
    (SELECT COUNT(*) FROM retail_sales_catalog.bronze.customers_raw WHERE CustomerID IS NOT NULL),
    (SELECT COUNT(*) FROM retail_sales_catalog.bronze.customers_raw) - 
    (SELECT COUNT(*) FROM retail_sales_catalog.bronze.customers_raw WHERE CustomerID IS NOT NULL)
UNION ALL
SELECT 
    'Silver Layer',
    'Clean Records',
    (SELECT COUNT(*) FROM retail_sales_catalog.silver.customers_clean),
    (SELECT COUNT(*) FROM retail_sales_catalog.bronze.customers_raw WHERE CustomerID IS NOT NULL) - 
    (SELECT COUNT(*) FROM retail_sales_catalog.silver.customers_clean)
UNION ALL
SELECT 
    'Gold Layer',
    'Active Customers (SCD)',
    (SELECT COUNT(*) FROM retail_sales_catalog.gold.dim_customer WHERE IsActive = 1),
    (SELECT COUNT(*) FROM retail_sales_catalog.silver.customers_clean) - 
    (SELECT COUNT(*) FROM retail_sales_catalog.gold.dim_customer WHERE IsActive = 1)
UNION ALL
SELECT 
    'Gold Layer',
    'Total Customers (All SCD)',
    (SELECT COUNT(*) FROM retail_sales_catalog.gold.dim_customer),
    (SELECT COUNT(*) FROM retail_sales_catalog.gold.dim_customer) - 
    (SELECT COUNT(*) FROM retail_sales_catalog.gold.dim_customer WHERE IsActive = 1)
ORDER BY 
    CASE stage 
        WHEN 'Bronze Layer' THEN 1 
        WHEN 'Silver Layer' THEN 2 
        WHEN 'Gold Layer' THEN 3 
    END,
    description;